# optimization_advisor\nGenerated from 00_common/optimization_advisor.py for Databricks notebook import.\n

In [ ]:
"""Query optimization and performance monitoring utilities."""\n\nfrom __future__ import annotations\n\nimport logging\nfrom dataclasses import dataclass\nfrom typing import Optional\n\nlogger = logging.getLogger(__name__)\n\n\n@dataclass\nclass OptimizationRecommendation:\n    """Represents a query optimization recommendation."""\n    recommendation_type: str\n    description: str\n    expected_improvement: str\n    implementation: str\n    priority: str  # HIGH, MEDIUM, LOW\n\n\nclass OptimizationAdvisor:\n    """\n    Analyzes table metadata and provides optimization recommendations.\n    \n    Recommendations include:\n    - Partitioning strategy\n    - Z-order clustering\n    - Data statistics\n    - Query patterns\n    """\n    \n    @staticmethod\n    def recommend_partitioning(\n        table_name: str,\n        column_cardinalities: dict[str, int],\n        total_rows: int,\n    ) -> Optional[OptimizationRecommendation]:\n        """\n        Recommend partitioning strategy.\n        \n        Args:\n            table_name: Table name\n            column_cardinalities: Dict of column -> cardinality\n            total_rows: Total row count\n            \n        Returns:\n            Optimization recommendation or None\n        """\n        # Find low-cardinality columns suitable for partitioning\n        low_card_cols = [\n            col for col, card in column_cardinalities.items()\n            if 2 <= card <= 1000  # Ideal partition cardinality range\n        ]\n        \n        if not low_card_cols:\n            return None\n        \n        primary_partition = low_card_cols[0]\n        return OptimizationRecommendation(\n            recommendation_type="PARTITIONING",\n            description=f"Add partitioning by {primary_partition}",\n            expected_improvement="2-5x faster queries on partitioned column",\n            implementation=f"""\nALTER TABLE {table_name}\nSET TBLPROPERTIES (\n  'delta.autoOptimize.optimizeWrite' = 'true',\n  'delta.autoOptimize.autoCompact' = 'true'\n);\n\nCREATE TABLE {table_name}_partitioned\nUSING DELTA\nPARTITIONED BY ({primary_partition})\nAS SELECT * FROM {table_name};\n\n-- Then swap tables\nALTER TABLE {table_name} RENAME TO {table_name}_old;\nALTER TABLE {table_name}_partitioned RENAME TO {table_name};\n""",\n            priority="HIGH",\n        )\n    \n    @staticmethod\n    def recommend_zorder(\n        table_name: str,\n        high_cardinality_columns: list[str],\n    ) -> Optional[OptimizationRecommendation]:\n        """\n        Recommend Z-order optimization.\n        \n        Args:\n            table_name: Table name\n            high_cardinality_columns: List of high-cardinality columns\n            \n        Returns:\n            Optimization recommendation or None\n        """\n        if not high_cardinality_columns:\n            return None\n        \n        zorder_cols = ",".join(high_cardinality_columns[:3])  # Z-order up to 3 columns\n        return OptimizationRecommendation(\n            recommendation_type="Z_ORDER",\n            description=f"Apply Z-order clustering on {zorder_cols}",\n            expected_improvement="2-3x faster on filter queries",\n            implementation=f"OPTIMIZE {table_name} ZORDER BY ({zorder_cols})",\n            priority="MEDIUM",\n        )\n    \n    @staticmethod\n    def recommend_statistics(\n        table_name: str,\n    ) -> OptimizationRecommendation:\n        """Recommend collecting table statistics."""\n        return OptimizationRecommendation(\n            recommendation_type="STATISTICS",\n            description="Collect and update table statistics",\n            expected_improvement="Better query planner decisions",\n            implementation=f"ANALYZE TABLE {table_name} COMPUTE STATISTICS",\n            priority="MEDIUM",\n        )\n\n\nclass PerformanceMonitor:\n    """\n    Monitors query execution times and identifies performance issues.\n    """\n    \n    def __init__(self, spark=None):\n        """Initialize PerformanceMonitor."""\n        self.spark = spark\n        self.metrics: list[dict] = []\n    \n    def record_execution(\n        self,\n        query_name: str,\n        duration_seconds: float,\n        rows_processed: int,\n        bytes_processed: int,\n    ) -> None:\n        """Record query execution metrics."""\n        throughput_mbps = (bytes_processed / (1024 ** 2)) / max(duration_seconds, 0.001)\n        self.metrics.append({\n            "query_name": query_name,\n            "duration_seconds": duration_seconds,\n            "rows_processed": rows_processed,\n            "bytes_processed": bytes_processed,\n            "throughput_mbps": throughput_mbps,\n        })\n        \n        if duration_seconds > 60:  # Log slow queries\n            logger.warning(f"Slow query detected: {query_name} took {duration_seconds:.1f}s")\n    \n    def get_slow_queries(self, threshold_seconds: float = 30.0) -> list[dict]:\n        """Get queries that exceeded duration threshold."""\n        return [\n            m for m in self.metrics\n            if m["duration_seconds"] > threshold_seconds\n        ]\n    \n    def write_metrics_to_table(\n        self,\n        catalog: str,\n        schema: str,\n        table_name: str,\n    ) -> int:\n        """Write performance metrics to Databricks table."""\n        if not self.spark or not self.metrics:\n            return 0\n        \n        try:\n            df = self.spark.createDataFrame(self.metrics)\n            table_fqn = f"{catalog}.{schema}.{table_name}"\n            df.write.mode("append").insertInto(table_fqn)\n            logger.info(f"Wrote {len(self.metrics)} performance metrics to {table_fqn}")\n            return len(self.metrics)\n        except Exception as e:\n            logger.error(f"Failed to write metrics: {e}")\n            return 0\n